# 🎯 Phase 3: Supervised Learning – Classification Models
### Course: Artificial Intelligence & Machine Learning (24CSE0316) | Continuous Evaluation
**Project: PlacementIQ AI — Student Placement Status Prediction (Placed vs. Not Placed)**

---
## 📑 Syllabus Topics Covered:
1. **Introduction to Classification:** Binary target categorization ($y \in \{0, 1\}$).
2. **Logistic Regression:** Log-odds modeling with Sigmoid activation function $\sigma(z) = \frac{1}{1 + e^{-z}}$.
3. **K-Nearest Neighbour (KNN):** Non-parametric instance-based voting via Euclidean distance.
4. **Decision Tree:** Recursive information gain / Gini impurity partitioning.
5. **Random Forest:** Ensemble bagging of decorrelated decision trees with feature randomness.
6. **Naïve Bayes Classifier:** Generative probabilistic classification using Bayes' theorem with feature conditional independence assumption:
   $$P(y|X) = \frac{P(y) \prod_{j=1}^d P(x_j|y)}{P(X)}$$
7. **Support Vector Machine (SVM):** Maximum margin hyperplane separation ($w^T x + b = 0$).
8. **Multi-Model Benchmark & Best Model Selection.**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", palette="muted")
print("✅ Classification libraries imported!")


In [ ]:
train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')

features = ['CGPA', 'Coding_Skills', 'Internships', 'Projects',
            'Communication_Skills', 'Aptitude_Test_Score', 'Soft_Skills_Rating',
            'Certifications', 'Backlogs']

# Standardize numerical features
scaler = StandardScaler()
X_train = scaler.fit_transform(train[features])
y_train = (train['Placement_Status'] == 'Placed').astype(int)

X_test = scaler.transform(test[features])
y_test = (test['Placement_Status'] == 'Placed').astype(int)

print(f"Training samples: {len(X_train):,} | Testing samples: {len(X_test):,}")


In [ ]:
# Define all 6 syllabus classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "K-Nearest Neighbour": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "Naïve Bayes": GaussianNB(),
    "Support Vector Machine": CalibratedClassifierCV(LinearSVC(random_state=42, max_iter=2000, dual='auto'), cv=3),
}

# Subsample test if needed for fast execution
results = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    
    results.append({
        "Classifier": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1_Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
    })

results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)
display(results_df)


In [ ]:
# Model Performance Comparison Bar Chart
melted = results_df.melt(id_vars="Classifier", value_vars=["Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC"])

plt.figure(figsize=(14, 6))
sns.barplot(data=melted, x="Classifier", y="value", hue="variable", palette="magma")
plt.title("Benchmarking All Syllabus Classification Models (80/20 Test Set)", fontweight='bold')
plt.ylabel("Score")
plt.ylim(0, 1.1)
plt.xticks(rotation=20)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()
